# Challenger Model Prototyping

Train a challenger model, log it to MLflow, then copy the `run_id` into the Control Panel (🧪 Challenger tab) to compare it against Production.

**Workflow:**
1. Load data from DB or parquet
2. Train your model
3. Choose a **primary metric** (the one the comparison & promotion decision is based on)
4. Log model + all metrics to MLflow with `mlflow.log_model(model, 'model')`
5. Copy `run_id` printed at the end
6. Paste into Control Panel → Challenger tab → select matching primary metric → trigger DAG

**Available primary metrics:** `roc_auc` · `pr_auc` · `f1` · `precision` · `recall` · `accuracy`

> The chosen metric is stored as a tag on the registered model version and shown in the Control Panel KPI card and Monitoring Dashboard.

In [1]:
import os
import pandas as pd
import numpy as np
import mlflow
from sqlalchemy import create_engine

# ── Connections (pre-wired from environment) ──────────────────────────────────
POSTGRES_HOST = os.getenv('POSTGRES_HOST', 'postgres')
POSTGRES_PORT = os.getenv('POSTGRES_PORT', '5432')
POSTGRES_DB   = os.getenv('POSTGRES_DB', 'mlops')
POSTGRES_USER = os.getenv('POSTGRES_USER', 'mlops_user')
POSTGRES_PASS = os.getenv('POSTGRES_PASSWORD', '')
MLFLOW_URI    = os.getenv('MLFLOW_TRACKING_URI', 'http://mlflow:5000')
MODEL_NAME    = os.getenv('MODEL_REGISTRY_NAME', 'credit-risk-classifier')

engine = create_engine(
    f'postgresql+psycopg2://{POSTGRES_USER}:{POSTGRES_PASS}@{POSTGRES_HOST}:{POSTGRES_PORT}/{POSTGRES_DB}'
)
mlflow.set_tracking_uri(MLFLOW_URI)
print('Connections ready.')

Connections ready.


In [2]:
# ── Load training data (all available records) ────────────────────────────────
FEATURE_COLUMNS = [
    'age', 'annual_income', 'credit_score', 'loan_amount',
    'loan_term_months', 'employment_length_years', 'home_ownership_encoded',
    'debt_to_income_ratio', 'num_credit_lines', 'payment_history_score',
]
TARGET = 'default_flag'

df = pd.read_sql(
    f"SELECT {', '.join(FEATURE_COLUMNS)}, {TARGET} FROM dwh_clean.cleaned_features "
    f"WHERE {TARGET} IS NOT NULL ORDER BY created_at",
    engine
)
print(f'Loaded {len(df):,} records  |  default rate: {df[TARGET].mean():.3f}')
df.head()

Loaded 14,285 records  |  default rate: 0.281


,age,annual_income,credit_score,loan_amount,loan_term_months,employment_length_years,home_ownership_encoded,debt_to_income_ratio,num_credit_lines,payment_history_score,default_flag
0,45.0,111804.976187,691.0,23434.501003,36.0,2.518965,0,0.2096,5.0,60.591740,1
1,29.0,84444.187534,677.0,11261.085533,48.0,10.034627,0,0.1334,8.0,95.456315,0
2,51.0,160021.544086,723.0,23399.479029,36.0,2.983461,2,0.1462,11.0,72.541014,0
3,53.0,27614.149507,710.0,11074.909895,48.0,6.185572,0,0.4011,8.0,79.195081,0
4,18.0,52434.494507,530.0,44896.820996,60.0,12.106974,2,0.8562,11.0,44.964962,1


In [3]:
# ── Data folder overview ──────────────────────────────────────────────────────
import glob

data_dirs = {
    'raw':         '/data/raw',
    'processed':   '/data/processed',
    'predictions': '/data/predictions',
    'monitoring':  '/data/monitoring',
}
for label, path in data_dirs.items():
    files = glob.glob(f'{path}/**/*.parquet', recursive=True)
    print(f'/data/{label:<12} {len(files):>3} parquet file(s)')

# ── Load processed parquet (example) ─────────────────────────────────────────
processed = glob.glob('/data/processed/*.parquet')
# df = pd.concat([pd.read_parquet(f) for f in processed], ignore_index=True)

# ── Load raw parquet (example) ────────────────────────────────────────────────
raw = glob.glob('/data/raw/*.parquet')
# df_raw = pd.concat([pd.read_parquet(f) for f in raw], ignore_index=True)

/data/raw            4 parquet file(s)
/data/processed      4 parquet file(s)
/data/predictions    3 parquet file(s)
/data/monitoring    12 parquet file(s)


In [4]:
# ── Train your challenger model ───────────────────────────────────────────────
import sys
sys.path.insert(0, '/opt/mlops/services')
from metrics import REGISTRY, compute_all_metrics

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier  # swap for anything you like

X = df[FEATURE_COLUMNS]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# ── Prototype your model here ─────────────────────────────────────────────────
model = RandomForestClassifier(
    n_estimators=200, max_depth=8, class_weight='balanced', random_state=42
)
model.fit(X_train, y_train)

y_prob = model.predict_proba(X_test)[:, 1]

# ── Compute all metrics at once ───────────────────────────────────────────────
test_metrics = compute_all_metrics(y_test.values, y_prob)

print('All supported metrics:')
for key, label in REGISTRY.items():
    print(f'  {label:<12} {test_metrics[key]:.4f}')

All supported metrics:
  ROC-AUC      0.8467
  PR-AUC       0.7036
  F1           0.6460
  Precision    0.5983
  Recall       0.7020
  Accuracy     0.7840


In [5]:
# ── Choose primary metric ─────────────────────────────────────────────────────
# This drives:
#   - the comparison decision in the Challenger DAG (challenger wins if this metric is higher)
#   - the KPI card label in Control Panel and Monitoring Dashboard
#   - the 'primary_metric' tag stored on the registered model version in MLflow
#
# Options: 'roc_auc' | 'pr_auc' | 'f1' | 'precision' | 'recall' | 'accuracy'
#
# When to pick something other than ROC-AUC:
#   pr_auc     — class imbalance is severe; ROC-AUC is too optimistic
#   f1         — false positives and false negatives matter equally, balanced
#   precision  — minimising false positives is critical (e.g. avoid wrongly denying good applicants)
#   recall     — minimising false negatives is critical (e.g. catch as many defaults as possible)
#   accuracy   — classes are balanced and raw correct-rate is the business target

PRIMARY_METRIC = 'f1'

In [6]:
# ── Log to MLflow and get run_id ──────────────────────────────────────────────
mlflow.set_experiment('challenger_experiments')

with mlflow.start_run(run_name=f'challenger_rf_{PRIMARY_METRIC}') as run:
    # Log all metrics so they appear in the MLflow UI
    mlflow.log_metrics({f'test_{k}': v for k, v in test_metrics.items()})

    # Tag the run with the chosen primary metric
    mlflow.set_tag('primary_metric', PRIMARY_METRIC)

    mlflow.log_params({
        'model_type':     'rf',
        'n_estimators':   200,
        'max_depth':      8,
        'train_size':     len(X_train),
        'primary_metric': PRIMARY_METRIC,
    })

    # IMPORTANT: log under artifact_path='model' — the DAG expects this path
    mlflow.sklearn.log_model(model, artifact_path='model')
    run_id = run.info.run_id

print()
print('=' * 60)
print(f'  run_id:        {run_id}')
print(f'  primary_metric: {REGISTRY[PRIMARY_METRIC]} ({PRIMARY_METRIC})')
print('=' * 60)
print()
print('Next steps:')
print('  1. Copy the run_id above')
print('  2. Open Control Panel → 🧪 Challenger tab')
print(f'  3. Paste run_id → set Primary metric to "{REGISTRY[PRIMARY_METRIC]}"')
print('  4. Click ▶ Run Challenger Comparison')

/usr/local/lib/python3.11/site-packages/_distutils_hack/__init__.py:53: UserWarning: Reliance on distutils from stdlib is deprecated. Users must rely on setuptools to provide the distutils module. Avoid importing distutils or import setuptools first, and avoid setting SETUPTOOLS_USE_DISTUTILS=stdlib. Register concerns at https://github.com/pypa/setuptools/issues/new?template=distutils-deprecation.yml
  warnings.warn(



  run_id:        1ddfd666e04e469196f35a336f63f9fb
  primary_metric: F1 (f1)

Next steps:
  1. Copy the run_id above
  2. Open Control Panel → 🧪 Challenger tab
  3. Paste run_id → set Primary metric to "F1"
  4. Click ▶ Run Challenger Comparison
